# Canonical MathLive rebuild showcase notebook — Phase 2

This notebook is the **primary evidence surface** for the Phase 2 rebuild.

> **Legacy filename note:** this notebook keeps the old path `examples/MathLive_identifier_system_showcase.ipynb` on purpose. The stable path is preserved so the visible verification surface stays easy to find.

Status for this notebook: **ready for user verification**.


## Phase goal

Phase 2 is trying to prove two visible things at once:

1. the generic `MathInput` field still behaves like a generic expression field,
2. a separate `IdentifierInput` field behaves more narrowly and follows an **explicit** identifier context policy.

### Implemented in this phase

- generic `MathInput` remains available as the baseline expression field
- separate `IdentifierInput` class
- explicit `context_policy` with two options:
  - `"context_only"`
  - `"context_or_new"`
- explicit plain-name contract for identifier values: empty or `[A-Za-z][A-Za-z0-9]*`
- explicit context suggestions shown in the identifier field
- more constrained identifier menu and shortcut surface

### Intentionally absent in this phase

- no role mutation inside `MathInput`
- no underscores in canonical identifier values yet
- no subscripts or superscripts as identifier structure
- no Greek-name policy
- no raw LaTeX identifier contract
- no broad semantic intelligence


## How to use this notebook

Run the cells in order.

For every visible demo below, the notebook tells you:

1. what this phase is trying to prove,
2. what should happen,
3. why that matters,
4. what would indicate the implementation is still wrong.

This notebook has **no saved outputs** and uses **no hidden browser automation**. Your manual sanity check is the main verification step.

### Optional but important console check

If you can open the browser console while running this notebook, watch for startup errors such as MathLive font-load failures or `Mathfield not mounted` failures. Those count against this phase.


In [1]:
import import_setup
from IPython.display import display
from gu_toolkit import IdentifierInput, MathInput


In [2]:
identifier_context = ["mass", "time", "speed"]
identifier_context


['mass', 'time', 'speed']

## Demo 1 — generic expression field baseline

### What is this demo trying to prove?
The rebuild still includes a generic expression field that is *not* an identifier-specific widget.

### What should happen?
After running the next cell, you should see a generic editable math field starting from `r"\sin(x)+1"`.

Open its visible menu using the hamburger button or by right-clicking in the field.

### Why should it happen?
Phase 2 needs a direct comparison point. The identifier field below should be more constrained than this generic baseline.

### What would indicate the implementation is still wrong?
Any of these count as failure:

- the generic field does not render,
- the field is not editable,
- the field does not show the constructor value,
- the menu is inaccessible.


In [3]:
generic_field = MathInput(value=r"\sin(x)+1")
generic_field


## Demo 2 — identifier field with `context_only`

### What is this demo trying to prove?
A separate identifier widget exists and is visibly more constrained than the generic expression field.

### Contract used in this phase
For `IdentifierInput`, a non-empty value must be a **plain identifier** matching `[A-Za-z][A-Za-z0-9]*`.

That means this phase intentionally does **not** support:

- `_` underscores,
- subscripts such as `x_1`,
- raw LaTeX such as `\mathrm{mass}`,
- expression input such as `x+1`.

### What should happen?
After running the next cell, you should see an identifier field initialized to `mass`, with the provided context shown as clickable suggestions.

Open the menu for this identifier field and compare it with the generic field above.

The identifier field menu should be **more constrained**. It should show only basic editing commands such as Undo, Redo, Cut, Copy, Paste, and Select all.

### What menu items should **not** appear in the identifier field?
The identifier field menu should **not** show expression-oriented items such as:

- solve / evaluate / simplify style actions,
- styling/background-selection actions,
- generic math insertion actions,
- function-oriented insertion actions.

### Why are those items inappropriate here?
This field is for entering **one identifier name**, not for editing a full math expression.

### What would indicate the implementation is still wrong?
Any of these count as failure:

- the identifier field renders exactly like the generic field with no visible constraint,
- the identifier field menu still exposes expression-heavy actions,
- the identifier field offers function insertion behavior that belongs to a generic expression editor.


In [9]:
context_only_field = IdentifierInput(
    value="mass",
    context_names=identifier_context,
    context_policy="context_only",
)
context_only_field
# BUG1: contract should be that of the robust identifier of gu_toolkit
# BUG2: upon entering an invalid name and leaving the widget, the widget should not be cleared but it should be in an "invalid" state (red outline).
# Remove line below with feedback and just keep the visual (aria enhanced) feedback mechanism (maybe with an optional red X or green check at the end of the field)
# HOWEVER DO KEEP THE FEEDBACK LINE BELOW THE identifier as a notebook showcase mechanism (separate from the code of IdentifierInput)
# BUG3: Does Mathlive not have an automatic suggestion mechanism. Use that instead of homebrewing
# BUG4: In the mathlive menu, allow insertion of context names. Mathlive has that mechanism. 


## Demo 3 — verify `context_only`

### What is this demo trying to prove?
`context_only` should accept only names from the provided context.

### The provided context
The allowed names for this demo are exactly:

- `mass`
- `time`
- `speed`

### What should happen?
Try both of these checks manually in the rendered field above:

1. Type `time`, then press Enter or click away from the field.
   - This **should** be accepted.
2. Type `angle`, then press Enter or click away from the field.
   - This **should not** be accepted under `context_only`.
   - The field should return to the last accepted value.

You can also click the visible suggestion buttons instead of typing.

### Why should it happen?
This proves the policy is explicit and visible: only the provided context is allowed.

### What would indicate the implementation is still wrong?
Any of these count as failure:

- `angle` is accepted in the `context_only` field,
- one of the provided names is rejected,
- the visible suggestions do not match the printed context.

Rerun the next cell after each manual edit to read the current Python value.


In [7]:
context_only_field.value


'mass'

## Demo 4 — identifier field with `context_or_new`

### What is this demo trying to prove?
The second explicit policy should still suggest context names while allowing a new plain identifier.

### What should happen?
After running the next cell, you should see another identifier field initialized to `mass`.

The same context names should be suggested.

Under `context_or_new`:

- typing `speed` should be accepted because it is in context,
- typing `angle` should also be accepted because it is a new plain identifier,
- typing `x+1`, `sin(x)`, `\sin`, or `theta_x` should **not** be accepted in this phase.

### Why should it happen?
This is the first role-aware UI behavior in the rebuild: the identifier field is narrower than a generic expression field, but the policy still decides whether unknown names are allowed.

### What would indicate the implementation is still wrong?
Any of these count as failure:

- `angle` is rejected even though the policy is `context_or_new`,
- expression-like input is accepted,
- the field quietly changes the rule without the explicit policy setting.


In [10]:
context_or_new_field = IdentifierInput(
    value="mass",
    context_names=identifier_context,
    context_policy="context_or_new",
)
context_or_new_field
# BUG5: Not all mathlive input helpers should be disabled here (as before). The exsting context should be there.

In [ ]:
context_or_new_field.value


**BUG6** The JS dev console reports this:
``` 
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Main-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Main-BoldItalic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Main-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Main-Italic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Math-Italic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Math-BoldItalic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_AMS-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Caligraphic-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Caligraphic-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Fraktur-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Fraktur-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_SansSerif-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_SansSerif-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_SansSerif-Italic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Script-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Typewriter-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Size1-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Size2-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Size3-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//fonts/KaTeX_Size4-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
mathlive.min.mjs:2979 MathLive 0.109.1: The math fonts could not be loaded from "https://cdn.jsdelivr.net/npm/mathlive//fonts" Object
nd @ mathlive.min.mjs:2979
widget.js:257 Error: Mathfield not mounted
    at set inlineShortcuts (mathlive.min.mjs:3233:11831)
    at Object.render (c331fc67-76ad-419a-960c-603af440341e:235:29)
    at widget.js:464:1
throw_anywidget_error @ widget.js:257
(anonymous) @ widget.js:480
widget.js:257 Error: Mathfield not mounted
    at set inlineShortcuts (mathlive.min.mjs:3233:11831)
    at Object.render (cdf33aae-ba53-4042-bdc9-99a38afbe91e:235:29)
    at widget.js:464:1
throw_anywidget_error @ widget.js:257
(anonymous) @ widget.js:480
mathlive.min.mjs:3233 Uncaught (in promise) Error: Mathfield not mounted
    at set inlineShortcuts (mathlive.min.mjs:3233:11831)
    at Object.render (c331fc67-76ad-419a-960c-603af440341e:235:29)
    at widget.js:464:1
set inlineShortcuts @ mathlive.min.mjs:3233
render @ c331fc67-76ad-419a-960c-603af440341e:235
(anonymous) @ widget.js:464
mathlive.min.mjs:3233 Uncaught (in promise) Error: Mathfield not mounted
    at set inlineShortcuts (mathlive.min.mjs:3233:11831)
    at Object.render (cdf33aae-ba53-4042-bdc9-99a38afbe91e:235:29)
    at widget.js:464:1
set inlineShortcuts @ mathlive.min.mjs:3233
render @ cdf33aae-ba53-4042-bdc9-99a38afbe91e:235
(anonymous) @ widget.js:464
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Starting WebSocket: ws://localhost:8888/api/kernels/2284b9f0-d143-407f-8f93-57e698ef79bb
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Kernel: restarting (2284b9f0-d143-407f-8f93-57e698ef79bb)
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Connection lost, reconnecting in 0 seconds.
_reconnect @ jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Starting WebSocket: ws://localhost:8888/api/kernels/2284b9f0-d143-407f-8f93-57e698ef79bb
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Keeping old LSP connection as the new kernel uses the same language
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Kernel: restarting (2284b9f0-d143-407f-8f93-57e698ef79bb)
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Connection lost, reconnecting in 0 seconds.
_reconnect @ jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Starting WebSocket: ws://localhost:8888/api/kernels/2284b9f0-d143-407f-8f93-57e698ef79bb
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Keeping old LSP connection as the new kernel uses the same language
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Starting WebSocket: ws://localhost:8888/api/kernels/2284b9f0-d143-407f-8f93-57e698ef79bb
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Kernel: restarting (2284b9f0-d143-407f-8f93-57e698ef79bb)
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Connection lost, reconnecting in 0 seconds.
_reconnect @ jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Starting WebSocket: ws://localhost:8888/api/kernels/2284b9f0-d143-407f-8f93-57e698ef79bb
jlab_core.4e90d6c30574b4474385.js?v=4e90d6c30574b4474385:1 Keeping old LSP connection as the new kernel uses the same language
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Main-BoldItalic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
mathlive.min.mjs:2979 MathLive 0.109.0: The math fonts could not be loaded from "https://cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts" Object
nd @ mathlive.min.mjs:2979
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Main-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Caligraphic-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Main-Italic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Main-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Math-Italic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Math-BoldItalic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Caligraphic-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_SansSerif-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Fraktur-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_SansSerif-Bold.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Size1-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Typewriter-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_SansSerif-Italic.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Size3-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Script-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Size4-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Fraktur-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_AMS-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive@0.109.0//fonts/KaTeX_Size2-Regular.woff2:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a status of 400 ()
cdn.jsdelivr.net/npm/mathlive//sounds/plonk.wav:1  Failed to load resource: the server responded with a 
```

## Manual verification checklist

Check these only after you have personally observed them in the notebook:

- [x] The generic `MathInput` field visibly renders and remains editable.
- [x] The generic field menu is accessible.
- [x] The `IdentifierInput` field visibly renders as a separate, more constrained widget.
- [x] The identifier field shows the provided context names as visible suggestions.
- [x] The identifier field menu does **not** show solve/evaluate/simplify, styling/background, or function-insertion style items.
- [x] In `context_only`, a provided name such as `time` is accepted.
- [x] In `context_only`, an unknown name such as `angle` is rejected and the field returns to the last accepted value.
- [x] In `context_or_new`, a new plain identifier such as `angle` is accepted.
- [x] In `context_or_new`, expression-like input such as `x+1` or `sin(x)` is rejected.
- [x] The visible suggestion list matches the printed context.
- [ ] The browser console does not report MathLive font-load or `Mathfield not mounted` errors while these demos render.

## Short report for this phase

### Implemented

- generic expression field retained
- separate identifier field added
- explicit context policies added
- visible context suggestions added
- identifier menu/shortcut surface constrained

### Not implemented yet

- broader identifier semantics
- underscore/subscript support
- raw LaTeX identifier contract
- broad semantic interpretation

### What you should test

Use the demos above to compare the menus, test the two explicit context policies, and try both allowed and rejected manual inputs.

### What would count as failure

- the identifier field menu still looks like a generic expression editor,
- `context_only` accepts unknown names,
- `context_or_new` rejects a valid new plain identifier,
- expression-like input is accepted as an identifier,
- startup errors appear in the browser console.

### What remains uncertain

This notebook does **not** prove future phases such as richer identifier syntax, semantic naming families, or compatibility with older hidden MathLive-era artifacts elsewhere in the repository.
